# Citation Metadata Matching — Prompts 08–18
**Project:** Exact vs. Fuzzy Metadata Matching for Citation Record Recovery Under Controlled Corruption

Run this notebook from top to bottom. It prepares the environment, downloads the public benchmark, validates its structure, runs three matching methods, saves real outputs, and creates artifacts for later paper/repository work.

**Research integrity:** do not fabricate outputs. The test metrics below are created only when the experiment code actually runs. Keep the downloaded output files as evidence.


## Prompt 08 — Environment configuration
This notebook uses CPU. A GPU is not needed for string matching.

In [ ]:
import os, sys, platform, json, subprocess, pathlib, hashlib
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())
print("CPU count:", os.cpu_count())
try:
    import psutil
    print("RAM (GB):", round(psutil.virtual_memory().total / (1024**3), 2))
except Exception as e:
    print("RAM diagnostic unavailable:", repr(e))


### Upload this project ZIP
Upload `citation_metadata_matching_project.zip` when prompted. The ZIP contains the scripts and notebook. This is a one-time upload per fresh Colab runtime.

In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
import zipfile, os, pathlib
zip_name = next((n for n in uploaded if n.endswith(".zip")), None)
if not zip_name:
    raise RuntimeError("Please upload citation_metadata_matching_project.zip")
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content")
PROJECT = "/content/citation_metadata_matching_project"
os.chdir(PROJECT)
print("Project directory:", os.getcwd())
print("Project files:", sorted(os.listdir(".")))


## Prompt 08 — Install and verify dependencies

In [ ]:
!pip -q install -r requirements.txt
import pandas, numpy, rapidfuzz, sklearn, matplotlib, requests
print("pandas", pandas.__version__)
print("numpy", numpy.__version__)
print("rapidfuzz", rapidfuzz.__version__)
print("scikit-learn", sklearn.__version__)
print("matplotlib", matplotlib.__version__)
print("requests", requests.__version__)
print("Dependency import check: PASS")


## Prompt 08 — Download and inspect the benchmark
The detector input is `corrupted_citation`. `original_citation` and labels are reserved for catalogue construction/evaluation only.

In [ ]:
import requests, json, hashlib, pathlib
DATA_URL = "https://raw.githubusercontent.com/FirstPrinciplesLabs/citecheck/main/dataset/corruption_metadata.json"
data_path = pathlib.Path("data/corruption_metadata.json")
r = requests.get(DATA_URL, timeout=90)
r.raise_for_status()
data_path.write_bytes(r.content)
dataset = json.loads(data_path.read_text())
print("SHA256:", hashlib.sha256(data_path.read_bytes()).hexdigest())
print("Collections:", len(dataset.get("collections", [])))
print("Citation count:", sum(len(c.get("citations", [])) for c in dataset.get("collections", [])))
print("Top-level keys:", list(dataset.keys()))
print("First collection keys:", list(dataset["collections"][0].keys()))
print("First citation keys:", list(dataset["collections"][0]["citations"][0].keys()))


## Prompt 09 — Build and sanity-check the baseline modules

In [ ]:
import sys
sys.path.insert(0, "src")
from matching import parse_citation, build_records
example = "[Smith et al., 2020, A Sample Paper](https://arxiv.org/pdf/2001.01234.pdf)"
print(parse_citation(example))
queries, catalogue, parse_failures = build_records(dataset)
print("Parseable queries:", len(queries))
print("Catalogue records:", len(catalogue))
print("Parse failures:", len(parse_failures))
assert len(queries) > 0 and len(catalogue) > 0
assert all(q["target_id"] in {c["record_id"] for c in catalogue} for q in queries)
print("Parser and target-integrity sanity checks: PASS")


## Prompt 10 — Run controlled experiment (development threshold selection + held-out test)
This cell executes M1 exact-field, M2 title fuzzy, and M3 weighted multi-field fuzzy matching. It saves the actual results in `results/`.

In [ ]:
!python scripts/run_experiment.py --data data/corruption_metadata.json --out results


## Prompt 10 — Inspect genuine outputs

In [ ]:
from pathlib import Path
import json, pandas as pd
print("Output files:", sorted(p.name for p in Path("results").glob("*")))
metrics = json.loads(Path("results/metrics.json").read_text())
print(json.dumps(metrics, indent=2))
pred = pd.read_csv("results/predictions.csv")
display(pred.head(10))


## Prompt 13 — Result visualization and deeper analysis

In [ ]:
from IPython.display import display, Image
display(Image("results/threshold_sensitivity.png"))
errors = pd.read_csv("results/error_analysis.csv")
print("Test-set error rows:", len(errors))
display(errors.head(20))


## Prompt 17 — Export repository evidence

In [ ]:
import shutil, os
archive = shutil.make_archive("/content/citation_metadata_matching_results", "zip", "results")
print("Created:", archive)


In [ ]:
from google.colab import files
files.download("/content/citation_metadata_matching_results.zip")


## Prompts 11, 12, 14–16, 18 — Evidence required before final writing
- Prompt 11: if any cell fails, copy the actual traceback and attach it here; do not invent a fix log.
- Prompt 12: confirm the comparison pathway using actual baseline outputs.
- Prompt 14: interpret the real metrics, errors and threshold sensitivity.
- Prompt 15: populate `paper/main.tex` only from verified outputs.
- Prompt 16: run a critical review after the draft is complete.
- Prompt 18: complete `docs/ai_assistance_log.md` with your real prompts, decisions and verification.
